# Semantic cache from scratch
Cache LLM answers by embedding similarity, sweep the threshold, and measure hits vs false hits.

## 1. Embedders

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
from data import INTENTS, all_queries, probe_set, query_stream, labeled_pairs
from embed import HashedEmbedder, TfidfEmbedder, features
corpus = [q for _, q in all_queries()]
hashed, tfidf = HashedEmbedder().fit(corpus), TfidfEmbedder().fit(corpus)
print(features('how do i cancel my order')[:8])
for a, b in [('how do i cancel my order', 'cancel my recent order'), ('how do i cancel my order', 'how do i track my order')]:
    print(f'{a!r} vs {b!r}: hashed={hashed.embed(a) @ hashed.embed(b):.3f} tfidf={tfidf.embed(a) @ tfidf.embed(b):.3f}')

## 2. The cache

In [ ]:
from cache import SemanticCache
c = SemanticCache(hashed, threshold=0.4, capacity=3, ttl=10)
c.put('how do i cancel my order', INTENTS['order_cancel']['answer'], now=0, intent='order_cancel', version=0)
for q in ['cancel my recent order', 'how do i track my order', 'is shipping free']:
    e, s = c.lookup(q, now=1)
    print(f'{q!r}: sim={s:.3f} ->', e.meta['intent'] if e else 'MISS')
print('after TTL:', c.lookup('cancel my recent order', now=20)[0], c.stats)

## 3. Threshold sweep on the probe set

In [ ]:
from simulate import probe_eval, auc, pair_sims
entries, probes = probe_set()
for t in [0.2, 0.3, 0.4, 0.5, 0.6]:
    r = probe_eval(entries, probes, hashed, t)
    print(f"t={t:.1f}  hit={r['hit_rate']:.3f}  false_hit={r['false_hit_rate']:.3f}")
s, y = pair_sims(labeled_pairs(42), hashed); print('pair AUC (hashed):', round(auc(s, y), 4))

## 4. Stream replay, eviction and savings

In [ ]:
from simulate import replay, exact_replay
stream = query_stream(2000, 42)
print('exact-match hit rate:', exact_replay(stream))
for cap in [4, 16, None]:
    r = replay(stream, hashed, 0.4, capacity=cap)
    print(f"capacity={cap}: hit={r['hit_rate']:.3f} false={r['false_hit_rate']:.3f} cost saving={r['cost_saving_pct']:.1f}%")

## 5. Full smoke run
Run `python run_smoke.py` from the repo root. It writes `results/`.